# Qwen3.5-4B → OpenVINO IR + HuggingFace Push

Runs the scripts in `qwen35/scripts/` end-to-end, exactly the same way
you would run them locally from a terminal.

| Step | Script called |
|------|---------------|
| 0 | Load Kaggle secrets |
| 1 | `git clone dsainvg/openvino-model-conv` |
| 2 | Install requirements |
| 3 | `pytest qwen35/tests/` *(toy smoke test, no weights needed)* |
| 4 | `qwen35/scripts/download_model.py` — pull weights from HF |
| 5 | `qwen35/scripts/convert_to_openvino.py` — ov.convert_model → IR |
| 6 | `qwen35/scripts/push_to_hf.py` — create HF repo + upload |

> **Kaggle Secrets needed** (Add-ons → Secrets):
> - `HF_TOKEN` — HuggingFace token with **write** scope
> - `HF_REPO_NAME` — target repo name *(default: `qwen35-4b-openvino-fp16`)*


## 0 · Secrets

In [1]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(key, fallback=None):
        try:
            return _s.get_secret(key)
        except Exception:
            return fallback
except ImportError:
    def _get(key, fallback=None):
        return os.environ.get(key, fallback)

HF_TOKEN     = _get("HF_TOKEN")
HF_REPO_NAME = _get("HF_REPO_NAME", "qwen35-4b-openvino-fp16")

if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN secret is missing. Add it under Add-ons → Secrets.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"HF_REPO_NAME : {HF_REPO_NAME}")
print("HF_TOKEN     : *** (set)")

HF_REPO_NAME : qwen35-4b-openvino-fp16
HF_TOKEN     : *** (set)


## 1 · Clone the converter repo

In [2]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/dsainvg/openvino-model-conv.git"
REPO_DIR = Path("/kaggle/working/openvino-model-conv")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)

QWEN35_DIR  = REPO_DIR / "qwen35"
SCRIPTS_DIR = QWEN35_DIR / "scripts"
MODEL_DIR   = Path("/kaggle/working/Qwen3.5-4B")
OUTPUT_DIR  = Path("/kaggle/working/ov_ir_qwen35_4b")

print(f"Repo    : {REPO_DIR}")
print(f"Scripts : {SCRIPTS_DIR}")

Updating a8546c5..ef7e2f4
Fast-forward
 qwen35/scripts/check_config.py        | 73 +++++++++++++++++++----------------
 qwen35/scripts/convert_to_openvino.py | 16 ++++++++
 2 files changed, 56 insertions(+), 33 deletions(-)
Repo    : /kaggle/working/openvino-model-conv
Scripts : /kaggle/working/openvino-model-conv/qwen35/scripts


From https://github.com/dsainvg/openvino-model-conv
   a8546c5..ef7e2f4  main       -> origin/main


## 2 · Install requirements

In [3]:
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("transformers", "openvino", "huggingface_hub", "safetensors",
    "sentencepiece", "tiktoken", "accelerate", "pytest")

print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.12.0+cpu which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.12.0+cpu which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/5

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 re

Done.


## 3 · Toy smoke test  (`qwen35/tests/`)

Runs against random-weight toy models — no downloads needed. Should finish in seconds.

In [4]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("Toy smoke tests FAILED — fix modeling code before converting real weights.")
print("\n✓ All toy tests passed.")

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/openvino-model-conv/qwen35
plugins: langsmith-0.7.30, anyio-4.13.0, typeguard-4.5.1
collecting ... collected 11 items

tests/test_end_to_end.py::test_convert_to_openvino_script FAILED         [  9%]
tests/test_toy_match.py::test_rmsnorm_zero_weight_acts_as_identity_on_unit_rms PASSED [ 18%]
tests/test_toy_match.py::test_rope_cache_shape PASSED                    [ 27%]
tests/test_toy_match.py::test_mlp_output_shape PASSED                    [ 36%]
tests/test_toy_match.py::test_attention_state_grows PASSED               [ 45%]
tests/test_toy_match.py::test_gated_deltanet_shapes PASSED               [ 54%]
tests/test_toy_match.py::test_gated_deltanet_conv_shift PASSED           [ 63%]
tests/test_toy_match.py::test_gated_deltanet_recurrent_state_changes PASSED [ 72%]
tests/test_toy

RuntimeError: Toy smoke tests FAILED — fix modeling code before converting real weights.

## 4 · Download Qwen3.5-4B weights  (`qwen35/scripts/download_model.py`)

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "download_model.py"),
        "--model",  "Qwen/Qwen3.5-4B",
        "--output", str(MODEL_DIR),
        "--token",  HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("download_model.py failed.")
print("\n✓ Model downloaded.")

## 5 · Convert to OpenVINO IR  (`qwen35/scripts/convert_to_openvino.py`)

Loads real BF16 weights, traces with `ov.convert_model`, saves `openvino_model.xml/.bin`.

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "convert_to_openvino.py"),
        "--model-dir", str(MODEL_DIR),
        "--output",    str(OUTPUT_DIR),
        "--dtype",     "bf16",
        # --compile-check omitted: Kaggle CPU may lack Intel-specific OV plugins
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("convert_to_openvino.py failed.")
print("\n✓ IR saved to", OUTPUT_DIR)

## 6 · Push to HuggingFace  (`qwen35/scripts/push_to_hf.py`)

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "push_to_hf.py"),
        "--ir-dir",    str(OUTPUT_DIR),
        "--repo-name", HF_REPO_NAME,
        "--token",     HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("push_to_hf.py failed.")